# Composite AiOIR: bounded smoke check
Enable Internet and GPU T4. Run All. No training, no full audit, no false→true switches. Downloads ~3.77 GB test ZIP plus checkpoints/repositories. Outputs: 3 discovery scenes × rain/low_haze/low_haze_snow × 2 models. Full-resolution FP32, batch 1. Runtime is not yet measured. Confirmation/holdout outputs are never opened. Download common_failure_smoke_results.zip after completion (also produced on model failure).

In [ ]:
import os, sys, subprocess, json, shutil, traceback
from pathlib import Path
REPO = Path('/kaggle/working/CoT-restoration')
URL = 'https://github.com/HoangKhanhTung0111/CoT-restoration.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(REPO)], check=True)
os.chdir(REPO)
WORK = Path('/kaggle/working/common_failure_smoke')
WORK.mkdir(exist_ok=True)
print('Project revision:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', 'gdown', 'einops', 'timm', 'fvcore', 'thop', 'scikit-image'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before running.'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Self-contained imports also prevent the earlier missing-subprocess error.
import os, sys, subprocess, json, shutil, traceback
from pathlib import Path
REPO = Path('/kaggle/working/CoT-restoration')
WORK = Path('/kaggle/working/common_failure_smoke')
BUNDLE = WORK / 'bundle'
BUNDLE.mkdir(parents=True, exist_ok=True)
errors = []
def run_logged(args, name):
    with (BUNDLE / (name + '.log')).open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'hybrid_cot_nafnet.common_failure_audit', *args, '--work', str(WORK)], cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        if process.wait() != 0:
            raise RuntimeError(name + ' failed; see log in result ZIP')
try:
    run_logged(['prepare'], 'prepare')
    for model in ['onerestore', 'mirage']:
        try:
            run_logged(['smoke', '--model', model, '--scenes', '3'], model)
        except Exception as exc:
            errors.append(str(exc))
except Exception:
    errors.append(traceback.format_exc())
finally:
    for name in ['manifest.json', 'sources.json']:
        if (WORK / name).exists(): shutil.copy2(WORK / name, BUNDLE / name)
    if (WORK / 'results').exists(): shutil.copytree(WORK / 'results', BUNDLE / 'results', dirs_exist_ok=True)
    (BUNDLE / 'run.json').write_text(json.dumps({'errors': errors, 'project_commit': subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip()}, indent=2))
    (BUNDLE / 'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))
    archive = shutil.make_archive('/kaggle/working/common_failure_smoke_results', 'zip', BUNDLE)
    print('Download:', archive)
if errors: raise RuntimeError('Smoke incomplete. Send the result ZIP; do not start a full run. ' + str(errors))
print('Technical smoke completed. Metrics are NOT a contribution result; panels need review.')
